# Age Regression CNN

## Imports

In [ ]:
import pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
from torchvision.transforms import v2
import numpy as np
import random
from torchvision.datasets import ImageFolder


import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
import dl_lib

from typing import Literal

### Set Random Seeds

In [ ]:
data_path = pathlib.Path("data")
SEED = 2537532
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.mps.manual_seed(SEED)

## Get Ages

In [ ]:
first_run = False
if first_run:
    dataset = ImageFolder(data_path.joinpath("Faces"))
    df = pd.DataFrame(dataset.imgs, columns=["Path", "DummyClass"])
    df["age"] = df["Path"].str.rpartition("/")[2].str.partition("_")[0]
    df.head()

Reorganize Dataset

In [ ]:
redo = False
if redo:
    for age in df["age"].unique():
        os.makedirs(data_path.joinpath(f"Cleaned/{age}"), exist_ok=True)

    cleaned_path = data_path.joinpath("Cleaned/")
    for row in df.values:
        path = pathlib.Path(row[0])
        shutil.move(path, cleaned_path.joinpath(f"{row[2]}/{path.name}"))

In [ ]:
dataset = ImageFolder(data_path.joinpath("Cleaned"), transform=v2.ToTensor())
train, sample_val, sample_test = data.random_split(
    dataset, [0.8, 0.1, 0.1], torch.Generator().manual_seed(SEED)
)

In [ ]:
train_idx = train.indices
val_idx = sample_val.indices
test_idx = sample_test.indices

del sample_val, sample_test

### Get mean, std

In [ ]:
mean, std = dl_lib.calc_mean_std(data.DataLoader(train))
with open("mean_std.json", "w") as file:
    file.write(pd.Series({"mean": mean.tolist(), "std": std.tolist()}).to_json())

In [ ]:
train_transforms = v2.Compose(
    [
        v2.RandomRotation((-45, 45)),
        v2.RandomVerticalFlip(),
        v2.RandomHorizontalFlip(),
        v2.RandomPerspective(),
        v2.RandomGrayscale(),
        v2.RandomInvert(),
        v2.GaussianBlur((3, 3)),
        v2.RandomApply([v2.JPEG((10, 80))]),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.GaussianNoise(),
        v2.Normalize(mean, std),
    ]
)

val_transform = v2.Compose(
    [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean, std)]
)

### Apply Transforms

In [ ]:
val = data.Subset(
    ImageFolder(data_path.joinpath("Cleaned"), transform=val_transform), val_idx
)
test = data.Subset(
    ImageFolder(data_path.joinpath("Cleaned"), transform=val_transform), test_idx
)
train = data.Subset(
    ImageFolder(data_path.joinpath("Cleaned"), transform=train_transforms), train_idx
)

In [ ]:
# Define CNN


class AgeCNN(nn.Module):
    def __init__(self, input_channels, hidden_dimensions=16):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv2d(input_channels, hidden_dimensions, (2, 2)),
            nn.BatchNorm2d(hidden_dimensions),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(hidden_dimensions, hidden_dimensions * 4, (2, 2)),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(hidden_dimensions * 4, hidden_dimensions * 8, (2, 2)),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.LazyLinear(hidden_dimensions * 16),
            nn.ReLU(),
            nn.Linear(hidden_dimensions * 16, hidden_dimensions * 8),
            nn.ReLU(),
            nn.Linear(hidden_dimensions * 8, hidden_dimensions * 4),
            nn.ReLU(),
            nn.Linear(hidden_dimensions * 4, 1),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
train_loader = data.DataLoader(train, batch_size=16, shuffle=True)
val_loader = data.DataLoader(val, batch_size=16)
test_loader = data.DataLoader(test, batch_size=16)

## Train Model

In [ ]:
device = dl_lib.pick_device()

model = AgeCNN(input_channels=3)
optimizer = optim.AdamW(model.parameters())
criterion = nn.L1Loss()

trained_CNN, losses = dl_lib.train_model(
    model, criterion, optimizer, train_loader, val_loader, 50, 4, device
)

In [ ]:
l1_loss = nn.L1Loss()
ys = []
preds = []
with torch.no_grad():
    trained_CNN.eval()
    val_loss = 0
    for X, y in val_loader:
        y = y.float()

        ys.append(y.numpy().ravel())
        X = X.to(device).float()
        preds.append(trained_CNN(X).cpu().numpy().ravel())

In [ ]:
ys = np.concatenate(ys)
preds = np.concatenate(preds)

In [ ]:
def MAE(pred, y):
    absolute_error = np.abs((pred - y))
    return np.mean(absolute_error)

In [ ]:
MAE(preds, ys)

In [ ]:
def int_model_prediction(model: torch.nn.Module, X) -> torch.Tensor:
    return torch.round(model(X))

In [ ]:
MAE(np.round(preds), ys)